# V08 — Visual Capstone: Full Analytical Dashboard

## Business Problem

**Company:** A UK-based online retailer with operations across 38 countries.

**Stakeholders:**
- **CFO:** Wants a revenue trend view with forecasts and anomaly flags
- **CMO:** Wants customer segmentation and behavior analysis
- **COO:** Wants geographic and product performance views
- **Board:** Wants a single-page executive summary

**Your deliverable:** A complete visual analytics package — a static multi-panel Plotly report AND an interactive Dash application, both telling the same story at different levels of detail.

---

**Rules:**
- Every chart must have a title, axis labels, and a one-sentence business interpretation in a markdown cell
- Every assert block must pass before moving to the next phase
- The Dash app must run inline (`jupyter_mode='inline'`)
- No chart should be a copy-paste — each must be built to a specific spec
- Corporate theme must be applied consistently throughout

**Reference:** [Plotly docs](https://plotly.com/python/) | [Dash docs](https://dash.plotly.com/)


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output, State
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# ── Load & clean dataset ──────────────────────────────────────────────────────
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail[retail['Quantity'] > 0].dropna(subset=['CustomerID'])
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['Month'] = retail['InvoiceDate'].dt.to_period('M').astype(str)
retail['Quarter'] = retail['InvoiceDate'].dt.to_period('Q').astype(str)
retail['DayOfWeek'] = retail['InvoiceDate'].dt.day_name()
retail['Hour'] = retail['InvoiceDate'].dt.hour
retail['Date'] = retail['InvoiceDate'].dt.date
retail['CustomerID'] = retail['CustomerID'].astype(int)
retail['IsReturn'] = retail['Quantity'] < 0

# ── Pre-computed aggregates ───────────────────────────────────────────────────
monthly = retail.groupby('Month').agg(
    Revenue=('Revenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Customers=('CustomerID','nunique')
).reset_index()
monthly['AvgOrderValue'] = (monthly['Revenue'] / monthly['Orders']).round(2)
monthly['RevenueGrowth'] = monthly['Revenue'].pct_change().round(4)

country_rev = (
    retail.groupby('Country')
    .agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'),
         Customers=('CustomerID','nunique'))
    .reset_index().sort_values('Revenue', ascending=False)
)

daily = (
    retail.groupby('Date')['Revenue'].sum()
    .reset_index().rename(columns={'Date':'date','Revenue':'revenue'})
)
daily['date'] = pd.to_datetime(daily['date'])
daily['dow'] = daily['date'].dt.dayofweek
daily['week'] = daily['date'].dt.isocalendar().week

# ── RFM customer features ─────────────────────────────────────────────────────
ref_date = retail['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = retail.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (ref_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()

# ── Country ISO mapping ───────────────────────────────────────────────────────
country_iso = {
    'United Kingdom':'GBR','Germany':'DEU','France':'FRA','EIRE':'IRL',
    'Spain':'ESP','Netherlands':'NLD','Belgium':'BEL','Switzerland':'CHE',
    'Portugal':'PRT','Australia':'AUS','Norway':'NOR','Sweden':'SWE',
    'Denmark':'DNK','Japan':'JPN','Finland':'FIN'
}
country_rev['ISO'] = country_rev['Country'].map(country_iso)

print(f"Retail: {retail.shape} | Monthly: {monthly.shape} | RFM: {rfm.shape}")
monthly.head(3)

---
## Phase 1 — Revenue Intelligence

**Audience: CFO.** Revenue trend, growth rates, anomaly detection, and a simple linear forecast.

### Phase 1.1 — Annotated Revenue Trend with Forecast

**Spec:** Build a single `go.Figure` combining:
1. **Actual revenue** bar chart (monthly) — color bars by MoM growth: green if positive, red if negative
2. **12-month rolling average** line overlay — color `'#1565C0'`, width 2, dashed
3. **Linear forecast** for 3 months ahead using `numpy.polyfit` on the time index — shown as a dashed line with a ±1.5σ shaded confidence band (`go.Scatter` with `fill='tonexty'`)
4. **Anomaly flags**: annotate months where revenue deviates > 1.5σ from rolling mean with a red triangle marker
5. Secondary y-axis: MoM growth rate as a `go.Scatter` line — color `'#FB8C00'`, reference line at y=0

- Title: `'Revenue Trend, Forecast & Anomalies'`
- Height: 500
- Apply corporate theme
- Assign to `fig_p1`

In [ ]:
# Compute required series
monthly['RollingMean'] = monthly['Revenue'].rolling(3, min_periods=1).mean()
monthly['RollingStd'] = monthly['Revenue'].rolling(3, min_periods=1).std().fillna(0)
monthly['IsAnomaly'] = abs(monthly['Revenue'] - monthly['RollingMean']) > 1.5 * monthly['RollingStd']

# Linear forecast
x_idx = np.arange(len(monthly))
coeffs = np.polyfit(x_idx, monthly['Revenue'], 1)
forecast_idx = np.arange(len(monthly), len(monthly) + 3)
forecast_rev = np.polyval(coeffs, forecast_idx)
forecast_months = [
    (pd.Period(monthly['Month'].iloc[-1], 'M') + i).strftime('%Y-%m')
    for i in range(1, 4)
]
residuals_std = np.std(monthly['Revenue'] - np.polyval(coeffs, x_idx))

bar_colors = [
    '#43A047' if g > 0 else '#E53935'
    for g in monthly['RevenueGrowth'].fillna(0)
]

# YOUR CODE HERE
fig_p1 = None

fig_p1.show()

In [ ]:
# --- ASSERTIONS ---
assert fig_p1 is not None
assert fig_p1.layout.title.text == 'Revenue Trend, Forecast & Anomalies'
assert fig_p1.layout.height == 500
assert fig_p1.layout.paper_bgcolor == '#FAFAFA', "Corporate theme not applied"
trace_types = [t.type for t in fig_p1.data]
assert 'bar' in trace_types, "Revenue bars missing"
assert 'scatter' in trace_types, "Trend/forecast lines missing"
# Secondary y-axis
assert fig_p1.layout.yaxis2 is not None, "Secondary y-axis missing"
# Forecast: at least one trace with future months
all_x = []
for t in fig_p1.data:
    if hasattr(t, 'x') and t.x is not None:
        all_x.extend(list(t.x))
assert any(m in str(all_x) for m in forecast_months), "Forecast months missing from chart"
# Anomaly markers
anomaly_traces = [t for t in fig_p1.data
                  if hasattr(t,'marker') and t.marker is not None
                  and getattr(t.marker,'symbol',None) is not None
                  and 'triangle' in str(t.marker.symbol)]
assert len(anomaly_traces) >= 1 or len(fig_p1.layout.annotations) >= 1, \
    "Anomaly markers or annotations required"
print("✓ Phase 1.1 passed")

**Business interpretation:** *(What is the revenue trajectory? Are anomalies seasonal or structural? Does the forecast suggest growth or decline?)*

### Phase 1.2 — Revenue Waterfall & Decomposition

**Spec:** Two-panel figure:
- Left: `go.Waterfall` showing MoM revenue changes (relative bars + final total bar)
  - Increasing: `'#26A69A'`, Decreasing: `'#EF5350'`, Total: `'#1565C0'`
  - Connector line: `'#90A4AE'`, width 1
  - Text on each bar: formatted `+£X,XXX` or `-£X,XXX`
- Right: Stacked area showing Revenue decomposed into UK vs Non-UK contribution over months
  - UK fill: `'rgba(21,101,192,0.6)'`, Non-UK fill: `'rgba(67,160,71,0.6)'`

- Use `make_subplots(cols=2, column_widths=[0.55, 0.45])`
- Title: `'Revenue Waterfall & Geographic Decomposition'`
- Height: 480
- Assign to `fig_p12`

In [ ]:
# Decompose UK vs Non-UK monthly revenue
uk_monthly = (
    retail[retail['Country']=='United Kingdom']
    .groupby('Month')['Revenue'].sum().reset_index()
    .rename(columns={'Revenue':'UK_Revenue'})
)
nonuk_monthly = (
    retail[retail['Country']!='United Kingdom']
    .groupby('Month')['Revenue'].sum().reset_index()
    .rename(columns={'Revenue':'NonUK_Revenue'})
)
geo_monthly = monthly[['Month','Revenue']].merge(uk_monthly, on='Month').merge(nonuk_monthly, on='Month')

mom_delta = monthly['Revenue'].diff().fillna(monthly['Revenue'].iloc[0])
measure = ['relative'] * (len(monthly)-1) + ['total']
wf_y = list(mom_delta.iloc[:-1]) + [monthly['Revenue'].iloc[-1]]
wf_text = [
    f"+£{v:,.0f}" if v >= 0 else f"-£{abs(v):,.0f}"
    for v in wf_y
]

# YOUR CODE HERE
fig_p12 = None

fig_p12.show()

In [ ]:
# --- ASSERTIONS ---
assert fig_p12.layout.title.text == 'Revenue Waterfall & Geographic Decomposition'
assert fig_p12.layout.height == 480
trace_types = [t.type for t in fig_p12.data]
assert 'waterfall' in trace_types, "Waterfall trace missing"
assert 'scatter' in trace_types, "Area chart traces missing"
wf = [t for t in fig_p12.data if t.type == 'waterfall'][0]
assert wf.measure[-1] == 'total', "Last waterfall bar must be 'total'"
assert wf.increasing.marker.color == '#26A69A'
assert wf.decreasing.marker.color == '#EF5350'
assert wf.text is not None and len(wf.text) == len(monthly)
# Both UK and Non-UK area traces
fill_traces = [t for t in fig_p12.data if hasattr(t,'fill') and t.fill in ('tozeroy','tonexty')]
assert len(fill_traces) >= 2, "Both UK and Non-UK area traces required"
print("✓ Phase 1.2 passed")

**Business interpretation:** *(Which months drove the biggest revenue swings? Is UK revenue growing faster or slower than international?)*

---
## Phase 2 — Customer Intelligence

**Audience: CMO.** Who are our customers, how do they behave, and how can we segment them?

### Phase 2.1 — RFM Segmentation Visualization

**Spec:**
1. Score each RFM dimension 1–4 using `pd.qcut`
   - Recency: lower = better (invert scoring: 4 = most recent)
   - Frequency and Monetary: higher = better
2. Run KMeans (k=4) on standardized RFM scores → `rfm['Cluster']`
3. Name clusters based on their mean RFM profile:
   - Highest monetary + high frequency → `'Champions'`
   - High recency + low frequency → `'New Customers'`
   - Low recency + high past monetary → `'At Risk'`
   - Lowest overall → `'Lost'`
   - Name programmatically — don't hardcode cluster numbers
4. Build 3-panel figure:
   - Panel 1: 3D scatter of R, F, M scores colored by cluster (`go.Scatter3d`)
   - Panel 2: Radar chart of mean R/F/M per cluster (`go.Scatterpolar`, 4 traces)
   - Panel 3: Bar chart of cluster sizes
5. Use `make_subplots` with `specs=[[{'type':'scene'},{'type':'polar'},{'type':'xy'}]]`
6. Title: `'RFM Customer Segmentation'`
7. Height: 550
8. Assign to `fig_p2`

In [ ]:
# Score RFM
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1,2,3,4]).astype(int)

# KMeans clustering
rfm_features = rfm[['R_Score','F_Score','M_Score']].values
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(rfm_scaled)

# Cluster profiling helper
cluster_profile = rfm.groupby('Cluster')[['R_Score','F_Score','M_Score','Monetary']].mean()
cluster_profile['Size'] = rfm.groupby('Cluster').size()

cluster_colors = ['#1565C0','#43A047','#FB8C00','#E53935']

# YOUR CODE HERE
# 1. Name clusters programmatically
# 2. Build fig_p2
rfm['SegmentName'] = rfm['Cluster'].map({})  # fill this
fig_p2 = None

fig_p2.show()

In [ ]:
# --- ASSERTIONS ---
assert fig_p2.layout.title.text == 'RFM Customer Segmentation'
assert fig_p2.layout.height == 550
trace_types = [t.type for t in fig_p2.data]
assert 'scatter3d' in trace_types, "3D scatter missing"
assert 'scatterpolar' in trace_types, "Radar chart missing"
assert 'bar' in trace_types, "Cluster size bar chart missing"
# 4 traces for 3D scatter (one per cluster)
s3d = [t for t in fig_p2.data if t.type == 'scatter3d']
assert len(s3d) == 4, "Need 4 scatter3d traces (one per cluster)"
# 4 radar traces
polar_traces = [t for t in fig_p2.data if t.type == 'scatterpolar']
assert len(polar_traces) == 4, "Need 4 radar traces"
# Segment names assigned
assert rfm['SegmentName'].nunique() == 4, "Must have 4 unique segment names"
expected_names = {'Champions', 'New Customers', 'At Risk', 'Lost'}
assert set(rfm['SegmentName'].unique()) == expected_names, \
    f"Segment names must be {expected_names}"
print("✓ Phase 2.1 passed")

**Business interpretation:** *(What % of customers are Champions vs Lost? What does the radar shape of At Risk customers tell you about how to re-engage them?)*

### Phase 2.2 — Customer Behavior Heatmap

**Spec:** Build a DOW × Hour revenue heatmap with marginal totals.

1. Pivot retail by `DayOfWeek` × `Hour`, summing Revenue
2. Build a `2×2` subplot figure:
   - Main panel (colspan=1, rowspan=1 — position [1,1]): `go.Heatmap` of DOW×Hour, colorscale `'YlOrRd'`
   - Right margin (position [1,2]): Horizontal bar — total revenue by DOW
   - Bottom margin (position [2,1]): Vertical bar — total revenue by Hour
   - Bottom-right (position [2,2]): `go.Indicator` showing peak hour
3. Annotate peak cell in heatmap (highest revenue DOW+Hour combination) with a star marker
4. DOW order: Monday–Sunday
5. Title: `'Revenue Activity Heatmap: Day × Hour'`
6. Height: 600
7. Assign to `fig_p22`

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_hour_pivot = (
    retail.groupby(['DayOfWeek','Hour'])['Revenue'].sum()
    .reset_index()
    .pivot(index='DayOfWeek', columns='Hour', values='Revenue')
    .reindex(day_order)
    .fillna(0)
)
peak_dow = dow_hour_pivot.max(axis=1).idxmax()
peak_hour = int(dow_hour_pivot.loc[peak_dow].idxmax())
dow_totals = dow_hour_pivot.sum(axis=1)
hour_totals = dow_hour_pivot.sum(axis=0)

# YOUR CODE HERE
fig_p22 = None

fig_p22.show()

In [ ]:
# --- ASSERTIONS ---
assert fig_p22.layout.title.text == 'Revenue Activity Heatmap: Day × Hour'
assert fig_p22.layout.height == 600
trace_types = [t.type for t in fig_p22.data]
assert 'heatmap' in trace_types, "Main heatmap missing"
assert 'bar' in trace_types, "Marginal bar charts missing"
assert 'indicator' in trace_types, "Peak hour indicator missing"
heatmap = [t for t in fig_p22.data if t.type == 'heatmap'][0]
# Check DOW order on y-axis
y_labels = list(heatmap.y)
assert y_labels == day_order, f"DOW must be Monday–Sunday, got {y_labels[:3]}..."
# Annotations (star on peak cell)
annotations = [a for a in fig_p22.layout.annotations if a.text and ('★' in a.text or 'peak' in a.text.lower() or '⭐' in a.text)]
assert len(annotations) >= 1 or any(
    hasattr(t,'marker') and t.marker is not None and 'star' in str(getattr(t.marker,'symbol',''))
    for t in fig_p22.data
), "Peak cell annotation or star marker required"
print(f"✓ Phase 2.2 passed — Peak: {peak_dow} at {peak_hour}:00")

**Business interpretation:** *(When should the business focus marketing spend? When is revenue lowest and why might that be?)*

---
## Phase 3 — Geographic & Product Intelligence

**Audience: COO.** Where are we selling, what products are driving growth, and how are operations performing?

### Phase 3.1 — Geographic Dashboard

**Spec:** 3-panel geographic view:
1. Left (geo type): `go.Choropleth` world map of revenue by country — `colorscale='Blues'`, `zmax=country_rev['Revenue'].quantile(0.9)`
2. Top-right (xy): Horizontal bar of top 12 countries, sorted ascending, colored by revenue gradient
3. Bottom-right (xy): Scatter of Revenue vs Orders by country (bubble sized by Customers), colored by continent group

- Continent groups: manually assign `'Europe'`, `'Other'` based on country list
- Use `make_subplots(rows=2, cols=2, specs=[[{'type':'geo','rowspan':2}, {'type':'xy'}],[None,{'type':'xy'}]])`
- Title: `'Global Revenue Intelligence'`
- Height: 600
- Assign to `fig_p3`

In [ ]:
european_countries = [
    'United Kingdom','Germany','France','EIRE','Spain','Netherlands',
    'Belgium','Switzerland','Portugal','Norway','Sweden','Denmark','Finland'
]
country_rev_mapped = country_rev[country_rev['ISO'].notna()].copy()
country_rev_mapped['Continent'] = country_rev_mapped['Country'].apply(
    lambda c: 'Europe' if c in european_countries else 'Other'
)
top12 = country_rev_mapped.nlargest(12, 'Revenue').sort_values('Revenue', ascending=True)

# YOUR CODE HERE
fig_p3 = None

fig_p3.show()

In [ ]:
# --- ASSERTIONS ---
assert fig_p3.layout.title.text == 'Global Revenue Intelligence'
assert fig_p3.layout.height == 600
trace_types = [t.type for t in fig_p3.data]
assert 'choropleth' in trace_types, "Choropleth map missing"
assert 'bar' in trace_types, "Country bar chart missing"
assert 'scatter' in trace_types, "Bubble scatter chart missing"
choro = [t for t in fig_p3.data if t.type == 'choropleth'][0]
assert choro.zmax is not None, "zmax must be set to cap color scale"
bar = [t for t in fig_p3.data if t.type == 'bar'][0]
assert bar.orientation == 'h', "Country bars must be horizontal"
print("✓ Phase 3.1 passed")

**Business interpretation:** *(How geographically concentrated is revenue? What risk does that present, and which markets show the most growth potential?)*

### Phase 3.2 — Product Hierarchy & Performance

**Spec:** Product intelligence view using hierarchical charts.

1. Identify top 50 products by revenue. For each, compute Revenue, Quantity, AvgUnitPrice, OrderCount.
2. Build 2-panel figure:
   - Left: `go.Treemap` — top 30 products, sized by Revenue, colored by AvgUnitPrice
     - `path=[px.Constant('All Products'), 'StockCode']`
     - `colorscale='RdYlGn'`, `branchvalues='total'`
   - Right: Scatter of Quantity vs Revenue for top 50 products
     - Sized by OrderCount, colored by AvgUnitPrice
     - Add quadrant lines at median Quantity and median Revenue
     - Label the top 5 by revenue with product code annotations
3. Use `make_subplots(cols=2, specs=[[{'type':'treemap'},{'type':'xy'}]])`
4. Title: `'Product Performance Analysis'`
5. Height: 520
6. Assign to `fig_p32`

In [ ]:
product_stats = (
    retail.groupby('StockCode').agg(
        Revenue=('Revenue','sum'),
        Quantity=('Quantity','sum'),
        AvgUnitPrice=('UnitPrice','mean'),
        OrderCount=('InvoiceNo','nunique')
    ).reset_index()
    .nlargest(50,'Revenue')
)
top30 = product_stats.nlargest(30,'Revenue')
med_qty = product_stats['Quantity'].median()
med_rev = product_stats['Revenue'].median()
top5_labels = product_stats.nlargest(5,'Revenue')['StockCode'].tolist()

# YOUR CODE HERE
fig_p32 = None

fig_p32.show()

In [ ]:
# --- ASSERTIONS ---
assert fig_p32.layout.title.text == 'Product Performance Analysis'
assert fig_p32.layout.height == 520
trace_types = [t.type for t in fig_p32.data]
assert 'treemap' in trace_types, "Treemap missing"
assert 'scatter' in trace_types, "Product scatter missing"
tm = [t for t in fig_p32.data if t.type == 'treemap'][0]
assert tm.branchvalues == 'total'
# Quadrant lines
shapes = fig_p32.layout.shapes
assert len(shapes) >= 2, "Must have at least 2 quadrant lines (median Revenue and Quantity)"
# Top 5 annotations
annots = [a.text for a in fig_p32.layout.annotations if a.text]
assert len(annots) >= 5, "Must annotate at least 5 top products"
print("✓ Phase 3.2 passed")

**Business interpretation:** *(Which products are high-volume/low-price vs low-volume/high-price? Where are the biggest revenue concentration risks?)*

---
## Phase 4 — Executive Summary Figure

**Audience: Board.** One figure that tells the complete story.

### Phase 4.1 — Single-Page Board Report

**Spec:** Build the complete board-level summary as a single `go.Figure` using `make_subplots`.

Layout (`rows=3, cols=3`):
- **Row 1, cols 1-3** (3 KPI indicators, `type='domain'` each): Total Revenue, Total Orders, Total Customers — with delta vs prior period
- **Row 2, cols 1-2** (`colspan=2`): Revenue trend line (monthly) with 3-month forecast, growth rate on secondary y-axis
- **Row 2, col 3**: Segment donut (Champions/New/At Risk/Lost from RFM)
- **Row 3, col 1**: Top 8 countries horizontal bar
- **Row 3, col 2**: DOW revenue bar (Mon–Sun)
- **Row 3, col 3**: Product revenue treemap (top 15)

Requirements:
- Title: `'UK Retail — Board Summary Dashboard'`
- Height: 950, width: 1200
- Full corporate theme throughout
- Every panel must have a subtitle annotation
- Assign to `fig_board`

In [ ]:
# Pre-compute board-level numbers
total_revenue = retail['Revenue'].sum()
total_orders = retail['InvoiceNo'].nunique()
total_customers = retail['CustomerID'].nunique()
prior_revenue = retail[retail['Month'] < monthly['Month'].iloc[-2]]['Revenue'].sum()

seg_counts = rfm['SegmentName'].value_counts()
top8_bar = country_rev.nlargest(8,'Revenue').sort_values('Revenue', ascending=True)

dow_rev = (
    retail.groupby('DayOfWeek')['Revenue'].sum()
    .reindex(day_order).reset_index()
)
top15_products = product_stats.nlargest(15,'Revenue')

# YOUR CODE HERE
fig_board = None

fig_board.show()

In [ ]:
# --- ASSERTIONS ---
assert fig_board is not None
assert fig_board.layout.title.text == 'UK Retail — Board Summary Dashboard'
assert fig_board.layout.height == 950
assert fig_board.layout.width == 1200
assert fig_board.layout.paper_bgcolor == '#FAFAFA', "Corporate theme required"
trace_types = [t.type for t in fig_board.data]
# All required chart types present
assert 'indicator' in trace_types, "KPI indicators missing"
assert trace_types.count('indicator') >= 3, "Need 3 KPI indicators"
assert 'scatter' in trace_types or 'bar' in trace_types, "Time series or bar missing"
assert 'pie' in trace_types, "Segment donut missing"
assert 'bar' in trace_types, "Country/DOW bar chart missing"
assert 'treemap' in trace_types, "Product treemap missing"
# Panel subtitles
annotations = [a.text for a in fig_board.layout.annotations if a.text]
assert len(annotations) >= 6, f"Need at least 6 panel annotations, got {len(annotations)}"
print(f"✓ Phase 4.1 passed — Board report complete with {len(fig_board.data)} traces")

**Business interpretation:** *(Write 3 bullet points — one key insight per stakeholder (CFO, CMO, COO) — that this dashboard supports.)*

- **CFO:** *...*
- **CMO:** *...*
- **COO:** *...*

---
## Phase 5 — Seaborn Statistical Annex

**Audience: Data Science team.** Statistical diagnostics that complement the Plotly dashboards.

### Phase 5.1 — Distribution & Normality Report

**Spec:** A static matplotlib/seaborn figure for the DS team's internal report.

Build a `2×3` subplot figure (`figsize=(16, 9)`):
1. Revenue per transaction — histogram + KDE (log scale)
2. Customer lifetime value (total Revenue per CustomerID) — histogram + KDE
3. Orders per customer — histogram
4. Revenue by customer segment (from RFM) — overlapping KDE per segment
5. Revenue by DOW — violin plots
6. Lorenz curve of customer revenue with Gini annotation

Requirements:
- `plt.style.use('seaborn-v0_8-whitegrid')` (or fallback)
- Consistent color palette: `sns.color_palette('Set2', 4)`
- Suptitle: `'Statistical Distribution Report — UK Retail'`
- Every panel has axis labels and title
- Assign to `fig_stat`

In [ ]:
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

palette = sns.color_palette('Set2', 4)

clv = retail.groupby('CustomerID')['Revenue'].sum()
orders_per_customer = retail.groupby('CustomerID')['InvoiceNo'].nunique()
rfm_seg = rfm.merge(
    rfm[['CustomerID','SegmentName']], on='CustomerID'
).drop_duplicates('CustomerID')
clv_seg = clv.reset_index().merge(
    rfm[['CustomerID','SegmentName']].drop_duplicates(), on='CustomerID'
)

sorted_clv = np.sort(clv.values)
lorenz_x = np.arange(1, len(sorted_clv)+1) / len(sorted_clv) * 100
lorenz_y = np.cumsum(sorted_clv) / sorted_clv.sum() * 100
gini = 1 - 2 * np.trapz(lorenz_y/100, lorenz_x/100)

# YOUR CODE HERE
fig_stat = None

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# --- ASSERTIONS ---
import matplotlib
assert isinstance(fig_stat, matplotlib.figure.Figure)
assert list(fig_stat.get_size_inches()) == [16.0, 9.0]
axes = fig_stat.get_axes()
assert len(axes) >= 6, f"Need 6 panels, got {len(axes)}"
suptitle = [t.get_text() for t in fig_stat.texts]
assert any('Statistical Distribution Report' in t for t in suptitle)
for ax in axes:
    assert ax.get_title() != '', f"All panels must have titles"
    assert ax.get_xlabel() != '' or ax.get_ylabel() != '', "All panels need axis labels"
# Gini annotation present somewhere
all_texts = [t.get_text() for ax in axes for t in ax.texts]
assert any('Gini' in t or 'gini' in t for t in all_texts), "Gini coefficient must be annotated"
print(f"✓ Phase 5.1 passed — Gini: {gini:.4f}")

---
## Phase 6 — Interactive Dash Application

**The full deliverable.** This app combines all phases into a single interactive dashboard.

### Phase 6.1 — Retail Intelligence Dash App

**Spec:** Build the complete interactive dashboard running inline in Jupyter.

**Layout:**
```
┌─────────────────────────────────────────────────────┐
│  HEADER: Title + Country Filter + Date Range        │
├──────────┬──────────┬──────────────────────────────┤
│ KPI: Rev │ KPI: Ord │ KPI: Customers               │
├──────────┴──────────┴──────────────────────────────┤
│  dcc.Tabs: [Revenue] [Customers] [Products] [Geo]   │
│  ┌─────────────────────────────────────────────┐   │
│  │  Tab content (changes per tab)              │   │
│  └─────────────────────────────────────────────┘   │
├─────────────────────────────────────────────────────┤
│  FOOTER: Last updated + data source                 │
└─────────────────────────────────────────────────────┘
```

**Tab contents:**
- **Revenue tab:** Trend chart (type=line/bar toggle via RadioItems) + Waterfall MoM chart
- **Customers tab:** RFM segment donut + Segment scatter (F vs M, sized by R)
- **Products tab:** Top-N product bar (slider for N=5..30) + Product treemap
- **Geography tab:** Choropleth map + Country bar chart

**Callbacks:**
1. Filters (Country + Date) → `dcc.Store` (cache filtered data)
2. Store + Tab → KPI values (3 outputs)
3. Store + Chart-type radio → Revenue tab figures
4. Store → Customer tab figures
5. Store + Top-N slider → Product tab figures
6. Store → Geography tab figures

**Styling:**
- Header: dark blue `#1565C0`, white text
- Cards: white background, `box-shadow: 0 2px 4px rgba(0,0,0,0.1)`
- Active tab: `border-bottom: 3px solid #1565C0`

- Port: `8080`

In [ ]:
from datetime import datetime

# Pre-compute everything needed by the app
all_countries = sorted(retail['Country'].unique())
date_min = retail['InvoiceDate'].min().date()
date_max = retail['InvoiceDate'].max().date()
segment_names = list(rfm['SegmentName'].unique())

HEADER_STYLE = {
    'backgroundColor': '#1565C0',
    'color': 'white',
    'padding': '16px 24px',
    'display': 'flex',
    'alignItems': 'center',
    'justifyContent': 'space-between'
}
CARD_STYLE = {
    'backgroundColor': 'white',
    'borderRadius': '8px',
    'padding': '16px',
    'boxShadow': '0 2px 4px rgba(0,0,0,0.1)',
    'margin': '8px'
}

# YOUR CODE HERE
app_final = Dash(__name__)
app_final.layout = None  # replace with full layout

# Define all 6 callbacks

app_final.run(jupyter_mode='inline', port=8080)

In [ ]:
# --- ASSERTIONS ---
layout_str = str(app_final.layout)

# Core layout elements
assert 'main-tabs' in layout_str or 'Tabs' in layout_str, "Tabs component missing"
assert 'tab-content' in layout_str or 'tab_content' in layout_str, "Tab content div missing"

# KPI outputs
for kpi in ['kpi-revenue', 'kpi-orders', 'kpi-customers']:
    assert kpi in layout_str, f"KPI component '{kpi}' missing"

# Filter controls
assert 'country' in layout_str.lower(), "Country filter missing"
assert 'date' in layout_str.lower(), "Date filter missing"

# Store component
assert 'Store' in layout_str or 'store' in layout_str, "dcc.Store missing"

# Callbacks registered
cb = app_final.callback_map
assert len(cb) >= 4, f"Expected >= 4 callbacks, got {len(cb)}"

# Store is an output (data caching pattern)
store_cb = [k for k in cb.keys() if 'store' in k.lower()]
assert len(store_cb) >= 1, "Store must be populated by a callback"

print(f"✓ Phase 6.1 passed — {len(cb)} callbacks, app running on port 8080")
print("Navigate between tabs and adjust filters to verify interactivity.")

---
## Capstone Self-Review Checklist

Before considering this capstone complete, verify:

**Charts:**
- [ ] Every chart has a title, axis labels, and a business interpretation
- [ ] Corporate theme applied consistently (paper_bgcolor, font, grid color)
- [ ] All assert blocks pass without errors
- [ ] No duplicate chart types across phases — each panel shows something unique

**Analysis:**
- [ ] RFM clusters are named programmatically (not hardcoded)
- [ ] Forecast is visible and labeled in Phase 1
- [ ] Anomalies are marked in the revenue trend
- [ ] Gini coefficient is annotated in Phase 5
- [ ] Board report (Phase 4) includes all 7 required panels

**Dash app:**
- [ ] App runs inline without errors
- [ ] Country + Date filters update all panels
- [ ] All 4 tabs render correct charts
- [ ] dcc.Store pattern used for data caching
- [ ] KPI tiles update on filter change

**Interpretation:**
- [ ] Phase 1 markdown: revenue trajectory identified
- [ ] Phase 2 markdown: segment action items stated
- [ ] Phase 3 markdown: geographic risk identified
- [ ] Phase 4 markdown: 3 stakeholder bullet points written

**This checklist mirrors what a senior data scientist or analytics lead would verify in a BCG X final-round case presentation.**